# Data Preprocessing Notebook

Clean, transform, and prepare data for analysis

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 1. Load Raw Data

In [ ]:
# Load raw CSV file
df = pd.read_csv('../dataset/raw_data/crop_prices.csv')
print(f'Original Data Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nFirst 5 rows:')
df.head()

## 2. Handle Missing Values

In [ ]:
# Check missing values before
print('Missing Values Before:')
print(df.isnull().sum())

# Fill missing values with mean (for numeric columns)
df = df.fillna(df.mean(numeric_only=True))

# Check missing values after
print('\nMissing Values After:')
print(df.isnull().sum())
print('\nMissing values handled')

## 3. Remove Duplicate Rows

In [ ]:
# Check duplicates before
print(f'Duplicate Rows Before: {df.duplicated().sum()}')

# Remove duplicates
df = df.drop_duplicates(keep='first')

# Check duplicates after
print(f'Duplicate Rows After: {df.duplicated().sum()}')
print(f'Data Shape After Removing Duplicates: {df.shape}')
print('Duplicates removed')

## 4. Convert Date Column to DateTime

In [ ]:
# Convert Date column to datetime format
print(f'Date Column Type Before: {df["Date"].dtype}')
df['Date'] = pd.to_datetime(df['Date'])
print(f'Date Column Type After: {df["Date"].dtype}')
print(f'Date Range: {df["Date"].min()} to {df["Date"].max()}')
print('Date column converted')

## 5. Handle Outliers using IQR Method

In [ ]:
# Function to handle outliers using Interquartile Range (IQR)
def handle_outliers(data, column, threshold=1.5):
    """
    Handle outliers by capping them at IQR bounds
    
    Args:
        data: DataFrame
        column: Column name to process
        threshold: IQR multiplier (1.5 is standard)
    """
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - (threshold * IQR)
    upper_bound = Q3 + (threshold * IQR)
    
    # Count outliers
    outliers = ((data[column] < lower_bound) | (data[column] > upper_bound)).sum()
    
    # Cap outliers
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
    
    return outliers

# Apply to numeric columns
print('Handling Outliers Using IQR Method (threshold=1.5)\n')
numeric_cols = ['Modal Price', 'Quantity']
for col in numeric_cols:
    outliers = handle_outliers(df, col)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    print(f'{col}:')
    print(f'  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {Q3-Q1:.2f}')
    print(f'  Outliers Handled: {outliers}')

print(f'\nData Shape After Outlier Handling: {df.shape}')
print('Outliers handled')

## 6. Label Encode Categorical Columns

In [ ]:
# Show unique commodities before encoding
print('Unique Commodities Before Encoding:')
print(df['Commodity'].unique())
print(f'Count: {df["Commodity"].nunique()}')

# Label encode Commodity column
le = LabelEncoder()
df['Commodity'] = le.fit_transform(df['Commodity'])

# Show mapping
print('\nEncoding Mapping (Commodity):')
mapping = dict(zip(le.transform(le.classes_), le.classes_))
for encoded, original in mapping.items():
    print(f'  {encoded} -> {original}')

print(f'\nUnique Values After Encoding: {df["Commodity"].unique()}')
print('Categorical columns encoded')

## 7. Final Data Summary

In [ ]:
# Display cleaned data summary
print('='*60)
print('CLEANED DATA SUMMARY')
print('='*60)
print(f'\nShape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nFirst 5 Rows:')
print(df.head())
print(f'\nStatistics:')
df.describe()

## 8. Save Cleaned Data

In [ ]:
import os

# Create output directory if not exists
output_dir = '../dataset/processed_data'
os.makedirs(output_dir, exist_ok=True)

# Save cleaned data
output_path = f'{output_dir}/cleaned_data.csv'
df.to_csv(output_path, index=False)

print(f'Cleaned data saved to {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.2f} KB')
print(f'\nNext: Run visualization.ipynb')